# Semana 1 — Fundamentos y datos del proyecto

En esta primera semana no entrenamos ningún modelo todavía. El objetivo es **conocer los datos de la planta de flotación**, aprender a cargarlos correctamente y construir una tabla ordenada (una fila por hora) que será el punto de partida de la Semana 2.

**Cómo está organizado este cuaderno**

- **Secciones 1 a 4 (obligatorias).** Casi todo el código está hecho. Tú completas las celdas marcadas con `# TODO`, que son de una o dos líneas; justo encima tienes siempre un ejemplo del mismo patrón o una pista.
- **Sección 5 (ampliación, opcional).** Para quien ya sepa Python o tenga tiempo: comprobaciones más finas sobre los datos, estadística y gráficas.

Ejecuta las celdas **en orden, de arriba abajo**, con `Mayús + Intro`. Si una construcción no te suena, búscala en `Semana1_Python_minimo.ipynb`.

> Cuaderno de **trabajo**. Intenta cada `TODO` con las pistas y el material de consulta. El profesor facilitará una referencia resuelta cuando corresponda.

## 0. Preparación

Cargamos las dos librerías que usaremos: **pandas** (tablas de datos) y **matplotlib** (gráficas). `FICHERO` es la ruta del CSV con los datos: **no la cambies**. Si al cargarlo sale `FileNotFoundError`, revisa la estructura de carpetas de la guía del entorno.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

FICHERO = '../datasets/dataset_flotacion_mini.csv'

## 1. Cargar el fichero (con cuidado)

Este CSV tiene una trampa: los números vienen con **coma decimal** (`55,2` en vez de `55.2`). Veamos qué pasa si lo cargamos sin avisar a pandas:

In [ ]:
# Carga ingenua (mal): pandas no sabe que la coma es el decimal
datos_texto = pd.read_csv(FICHERO)
datos_texto.head()

Fíjate en los números: aparecen con coma (`55,2`). Para pandas eso es **texto**, no un número, y con texto no se puede calcular ni una media. Lo arreglamos indicando `decimal=','`:

In [ ]:
# TODO: carga el fichero indicando que la coma es el separador decimal.
#       Es la misma orden de la celda anterior, añadiendo decimal=',' dentro del paréntesis.
#       pista: pd.read_csv(FICHERO, decimal=',')
datos = ...
datos.head()

Ahora los números salen con punto: pandas ya los entiende como números. Miramos el tamaño de la tabla (`shape`: filas y columnas) y si falta algún valor (`isna`):

In [ ]:
print('Filas y columnas:', datos.shape)
print('Valores ausentes:', datos.isna().sum().sum())

**Pregunta:** ¿cuántas filas y columnas tiene el fichero? ¿Falta algún valor? Mira los nombres de las columnas en la salida de `datos.head()`: ¿cuáles describen el mineral que entra, cuáles el proceso y cuáles la calidad del producto que sale?

## 2. La columna `date` y el muestreo mixto

La columna `date` (fecha y hora) se ha cargado como texto. La convertimos a tipo fecha con `pd.to_datetime`, para que pandas pueda ordenar y comparar instantes:

In [ ]:
# TODO: convierte la columna 'date' a tipo fecha.
#       pista: datos['date'] = pd.to_datetime(datos['date'])

datos['date'].head(10)

**Un detalle importante: `date` solo llega a la hora, no a los segundos.** Verás la misma fecha repetida muchas veces seguidas. No es un error: las variables de proceso se registran cada 20 s, pero la marca de tiempo solo llega a la hora, así que **todas las medidas de una misma hora comparten el mismo `date`**.

Quedémonos con las filas de una hora concreta. Es un **filtro**, como los de la sección 7 de *Python mínimo*:

In [ ]:
una_hora = datos[datos['date'] == '2017-03-10 02:00:00']
print('Filas en esa hora:', len(una_hora))

`nunique()` cuenta cuántos valores **distintos** hay en una columna. Para la sílice del concentrado, dentro de esa hora:

In [ ]:
print('Valores distintos de sílice en esa hora:', una_hora['% Silica Concentrate'].nunique())

In [ ]:
# TODO: haz lo mismo para la columna 'Amina Flow'
#       (copia la línea de la celda anterior y cambia el nombre de la columna)


**Pregunta:** ¿qué observas? Una de las dos variables es constante dentro de la hora y la otra no. ¿Por qué crees que es así? (Piensa en cómo se mide cada una: ¿un sensor en la planta o un análisis de laboratorio?)

Veámoslo en una gráfica: las ~180 medidas de amina dentro de esa hora.

In [ ]:
una_hora['Amina Flow'].plot(figsize=(7, 3))
plt.xlabel('fila del fichero (una cada 20 s)')
plt.title('Amina dentro de una hora')
plt.show()

In [ ]:
# TODO: dibuja igual la columna '% Silica Concentrate' dentro de la misma hora
#       (copia la celda anterior y cambia la columna y el título). ¿Qué forma tiene?


Por último, ¿cuántas filas tiene cada hora? `value_counts()` cuenta cuántas veces aparece cada valor de `date`:

In [ ]:
filas_por_hora = datos['date'].value_counts()
print('Horas distintas:', len(filas_por_hora))
print('Máximo de filas en una hora:', filas_por_hora.max())
print('Mínimo de filas en una hora:', filas_por_hora.min())

**Pregunta:** una hora completa debería tener 180 filas (3.600 s / 20 s). ¿Las tienen todas? ¿Cuántas horas distintas hay?

## 3. Explorar la señal objetivo (% de sílice)

La sílice del concentrado es la **impureza a minimizar** y lo que querremos predecir a partir de la Semana 2. `describe()` da de una vez sus estadísticos básicos: número de valores (`count`), media (`mean`), desviación típica (`std`), mínimo, cuartiles (`25%`, `50%` = mediana, `75%`) y máximo. Ejemplo con la amina:

In [ ]:
datos['Amina Flow'].describe()

In [ ]:
# TODO: estadísticos básicos de '% Silica Concentrate' (mismo patrón que la celda anterior)


**Pregunta:** ¿entre qué valores se mueve la sílice del concentrado? ¿Cuál es la media y cuál la mediana? ¿Se parecen?

## 4. Enventanado: de 20 segundos a una tabla por hora

Aquí está el trabajo central de la semana: **resumir las ~180 medidas de cada hora en una sola fila**. Lo hacemos en cuatro pasos:

1. Juntar las 7 columnas de aire en una y las 7 de nivel en otra.
2. Calcular la media de cada hora.
3. Montar la tabla con las columnas que nos interesan.
4. Añadir una etiqueta de calidad y guardar.

### 4.1 Aire y nivel medios

La planta tiene 7 **columnas de flotación**: los depósitos verticales donde se separa la sílice. De cada una se mide el caudal de aire y el nivel, así que en la tabla de datos son 14 columnas (`Flotation Column 01 Air Flow`, …, `Flotation Column 07 Level`). En cambio, la sílice y el hierro del concentrado son **un único análisis** del producto final, el que sale de la planta.

El dataset no indica cómo están conectadas las 7 columnas de flotación entre sí (en serie, en paralelo…), así que las tratamos en conjunto: resumimos las 14 columnas de datos en una media de aire y una de nivel. Es una simplificación: si una columna de flotación se comportara distinto que las demás, la media lo ocultaría.

`mean(axis=1)` calcula la media **de cada fila** (entre columnas de datos), en lugar de la media de cada columna.

In [ ]:
cols_aire = ['Flotation Column 01 Air Flow', 'Flotation Column 02 Air Flow',
             'Flotation Column 03 Air Flow', 'Flotation Column 04 Air Flow',
             'Flotation Column 05 Air Flow', 'Flotation Column 06 Air Flow',
             'Flotation Column 07 Air Flow']
cols_nivel = ['Flotation Column 01 Level', 'Flotation Column 02 Level',
              'Flotation Column 03 Level', 'Flotation Column 04 Level',
              'Flotation Column 05 Level', 'Flotation Column 06 Level',
              'Flotation Column 07 Level']

datos['aire_medio'] = datos[cols_aire].mean(axis=1)

# TODO: crea la columna 'nivel_medio' igual que 'aire_medio', pero con cols_nivel


### 4.2 La media de cada hora: `groupby`

`groupby('date')` junta las filas que tienen la misma hora, y `.mean()` calcula la media de cada columna dentro de cada grupo. El resultado tiene **una fila por hora**: es lo que viste en la sección 7 de *Python mínimo* con la tabla pequeña.

Con `as_index=False` la hora se queda como una **columna normal** (`date`), y las filas se numeran 0, 1, 2… como en cualquier tabla.

In [ ]:
medias = datos.groupby('date', as_index=False).mean()
print('Filas:', len(medias))
medias.head()

Dos resúmenes más con el mismo `groupby`:

- Las columnas de **alimentación** (`% Iron Feed`, `% Silica Feed`) son análisis de laboratorio y tienen el mismo valor en toda la hora. Para ellas no hace falta la media: nos quedamos con el **primer** valor de cada hora, con `.first()`.
- `.size()` cuenta cuántas filas tiene cada hora (en una columna llamada `size`).

Como los tres resúmenes salen del mismo `groupby`, tienen las mismas 112 horas **en el mismo orden**: la fila 0 es la misma hora en `medias`, en `primeros` y en `filas`. Por eso podremos juntarlos en la misma tabla.

In [ ]:
primeros = datos.groupby('date', as_index=False).first()
filas = datos.groupby('date', as_index=False).size()
filas.head()

### 4.3 Montar la tabla

Creamos una tabla vacía y la llenamos **columna a columna**, con nombres cortos en castellano. Todas las líneas siguen el mismo patrón:

```python
tabla['nombre_nuevo'] = medias['nombre en el fichero']
```

In [ ]:
tabla = pd.DataFrame()
tabla['date']          = medias['date']                  # la hora
tabla['n_muestras']    = filas['size']                   # nº de medidas en la hora
tabla['silice']        = medias['% Silica Concentrate']  # OBJETIVO: lo que querremos predecir
tabla['iron_feed']     = primeros['% Iron Feed']
tabla['silica_feed']   = primeros['% Silica Feed']
tabla['almidon_medio'] = medias['Starch Flow']

# TODO: añade el resto de columnas, todas desde 'medias' y en este orden:
#   'amina_media'  <-  'Amina Flow'
#   'caudal_pulpa' <-  'Ore Pulp Flow'
#   'ph_medio'     <-  'Ore Pulp pH'
#   'densidad'     <-  'Ore Pulp Density'
#   'aire_medio'   <-  'aire_medio'
#   'nivel_medio'  <-  'nivel_medio'


tabla.head()

**Importante — disciplina para no hacer trampa.** Fíjate en que, aparte del objetivo (`silice`), la tabla solo tiene **variables de entrada y de proceso**. No hemos metido `% Iron Concentrate`: el hierro del concentrado se mide en el laboratorio a la vez que la sílice, así que usarlo para predecir la sílice sería hacer trampa (en el curso lo llamaremos **fuga de información**). Esta disciplina nos acompañará todo el curso.

### 4.4 Etiqueta de calidad y guardar

Añadimos una etiqueta que usaremos en la Semana 3: **fuera de especificación** (`1`) si la sílice supera el 3,0 %, y `0` si no. Es el patrón de la sección 7 de *Python mínimo*: una comparación sobre toda la columna, convertida en 1/0 con `.astype(int)`.

In [ ]:
UMBRAL_SILICE = 3.0

# TODO: crea la columna 'fuera_spec'.
#       pista: tabla['fuera_spec'] = (tabla['silice'] > UMBRAL_SILICE).astype(int)


tabla['fuera_spec'].value_counts()

Hemos pasado de **20.000 medidas** a una tabla de **una fila por hora**. **Esta tabla es el punto de partida de la Semana 2.** Guárdala. `index=False` evita que se guarde también la numeración de las filas (0, 1, 2…), que no es un dato:

In [ ]:
# TODO: guarda la tabla en '../datasets/flotacion_horaria_mini.csv', sin la numeración de filas.
#       pista: tabla.to_csv('../datasets/flotacion_horaria_mini.csv', index=False)


### 4.5 Comprobación final

Si todo está bien, verás **112 filas y 13 columnas**, una sílice media de **2,65 %** y **38 horas** fuera de especificación.

In [ ]:
print('Filas y columnas:', tabla.shape)
print('Sílice media:', round(tabla['silice'].mean(), 2))
print('Horas fuera de especificación:', tabla['fuera_spec'].sum())

Y para terminar, la sílice hora a hora a lo largo de los 5 días:

In [ ]:
tabla.plot(x='date', y='silice', figsize=(8, 3), legend=False)
plt.axhline(UMBRAL_SILICE, color='red', linestyle='--')
plt.ylabel('% sílice en el concentrado')
plt.title('Sílice hora a hora (línea roja: límite de especificación)')
plt.show()

### Resumen de la semana

- Cargamos el CSV con `decimal=','` (si no, los números son texto).
- Entendimos el **muestreo mixto**: proceso cada 20 s, laboratorio una vez por hora, y la fecha se repite dentro de cada hora.
- Vimos que hay **horas incompletas** (no todas tienen 180 filas).
- Construimos `datasets/flotacion_horaria_mini.csv`: **una fila por hora**, con variables de proceso, el objetivo (`silice`) y una etiqueta de calidad (`fuera_spec`), **sin fugas de información**.

Con esa tabla arrancaremos la Semana 2. Si te sobra tiempo, sigue con la ampliación.

## 5. Ampliación (opcional)

Para quien ya sepa Python o tenga tiempo. Aquí hay menos ayuda: cada apartado plantea una pregunta y da una pista, y tú decides cómo resolverlo. Los apartados son independientes y ninguno modifica la tabla que ya has guardado.

### 5a. ¿Valores físicamente plausibles?

Una comprobación barata antes de fiarse de unos datos: porcentajes entre 0 y 100, pH entre 0 y 14, caudales y densidades no negativos. No sustituye a mirar los datos con calma, pero atrapa errores burdos.

In [ ]:
rangos_esperados = {
    '% Iron Feed': (0, 100), '% Silica Feed': (0, 100),
    '% Iron Concentrate': (0, 100), '% Silica Concentrate': (0, 100),
    'Ore Pulp pH': (0, 14), 'Ore Pulp Density': (0, None),
    'Starch Flow': (0, None), 'Amina Flow': (0, None), 'Ore Pulp Flow': (0, None),
}
# TODO: recorre el diccionario y, para cada columna, compara su mínimo y su máximo
#       con el rango esperado (None = sin límite). Imprime 'OK' o 'REVISAR'.
#       pista: for col, (lo, hi) in rangos_esperados.items(): ...


### 5b. ¿Es la sílice constante en *todas* las horas?

En la sección 2 vimos que la sílice era constante en **una** hora. No lo demos por sentado en las 112: para cada una de las 4 columnas de laboratorio, cuenta en cuántas horas hay más de un valor distinto.

In [ ]:
# TODO: para cada columna de la lista, cuenta en cuántas horas hay más de un valor distinto.
#       pista: datos.groupby('date')[col].nunique() da el nº de valores distintos en cada hora
for col in ['% Silica Concentrate', '% Iron Concentrate', '% Iron Feed', '% Silica Feed']:
    ...


Si te ha salido alguna hora donde la sílice del concentrado **no** es constante, búscala y dibuja la sílice dentro de esa hora.

In [ ]:
# TODO: localiza la(s) hora(s) con más de un valor de sílice y dibuja la sílice dentro de ella.
#       pista: datos.groupby('date')['% Silica Concentrate'].agg(['nunique', 'min', 'max'])


**Pregunta:** ¿qué forma tiene la sílice dentro de esa hora? ¿Por qué crees que en la sección 4 usamos `.mean()` para la sílice y `.first()` solo para las columnas de alimentación?

### 5c. ¿Hay huecos en el eje temporal?

¿Están todas las horas seguidas o falta alguna? Es la primera comprobación que conviene hacer sobre cualquier serie temporal nueva. Calcula la diferencia entre cada hora y la anterior y cuenta cuántas veces aparece cada diferencia.

In [ ]:
# TODO: diferencia entre cada hora y la anterior, y cuántas veces aparece cada diferencia.
#       pista: pd.Series(sorted(datos['date'].unique())), y después .diff().value_counts()


**Pregunta:** ¿hay huecos? El mini-dataset cubre solo 5 días seguidos; ¿crees que pasará lo mismo con los 6 meses de datos que usarás desde la Semana 2?

### 5d. Horas incompletas

Cuenta cuántas horas de `tabla` están completas (`n_muestras == 180`) y cuántas no, y crea una tabla `completas` solo con las completas.

In [ ]:
# TODO: cuántas horas tienen n_muestras == 180 y cuántas no; crea la tabla 'completas'.


### 5e. Reconstruir la marca de tiempo de 20 s

`date` no distingue las filas dentro de una hora. Reconstruye el instante real sumando 0, 20, 40… segundos a cada fila de su hora, en una columna nueva `timestamp`.

In [ ]:
# TODO: crea 'seg' = posición de la fila dentro de su hora * 20, y 'timestamp' = date + esos segundos.
#       pista: datos.groupby('date').cumcount() y pd.to_timedelta(..., unit='s')


**Pregunta:** ¿es exacta esta reconstrucción en las horas incompletas?

### 5f. Estudio estadístico de la tabla horaria

Ahora que tienes una fila por hora, explora la tabla:

1. Estadísticos de todas las columnas (`describe`). ¿Qué variables cambian más de una hora a otra?
2. Distribución de la sílice horaria: histograma y diagrama de caja. ¿Es simétrica? ¿Hay valores atípicos?
3. Correlación de cada variable con la sílice (`corr`). ¿Cuál se relaciona más?
4. Diagrama de dispersión de esa variable frente a la sílice.
5. Media de cada variable en las horas dentro y fuera de especificación (`groupby('fuera_spec')`).

In [ ]:
# TODO 1: estadísticos de todas las columnas de la tabla


In [ ]:
# TODO 2: histograma y diagrama de caja de tabla['silice'] (plt.hist, plt.boxplot)


In [ ]:
# TODO 3: correlación de cada variable de proceso con 'silice'.
#         Quita antes las columnas que no son variables de proceso: date, n_muestras, fuera_spec.


In [ ]:
# TODO 4: diagrama de dispersión de la variable más correlacionada frente a 'silice'


In [ ]:
# TODO 5: media de cada variable en las horas dentro (0) y fuera (1) de especificación


**Pregunta:** ¿qué variables parecen distinguir las horas buenas de las malas? ¿Te fiarías de estas conclusiones con solo 112 horas? Correlación no es causalidad: ¿qué otra cosa podría explicar una correlación fuerte?

### 5g. La forma compacta: `.agg()` con nombres

Quien ya conozca pandas puede construir toda la tabla de la sección 4.3 en **una sola instrucción**, con `groupby(...).agg(...)` y una pareja `(columna, operación)` para cada columna nueva (sección 7 de *Python mínimo*). Hazlo y comprueba que sale exactamente igual que `tabla`.

In [ ]:
# TODO: construye 'tabla_agg' con groupby('date', as_index=False).agg(...) y compárala con
#       tabla.drop(columns='fuera_spec') usando .equals(...)
#       pista: n_muestras=('% Silica Concentrate', 'size'), silice=('% Silica Concentrate', 'mean'), ...


### 5h. ¿Es demasiado simple la tabla horaria?

La tabla de la sección 4 resume cada hora con **medias** y mezcla las 7 columnas de flotación en una media de aire. Eso pierde dos cosas: la **variabilidad dentro de la hora** (una hora estable y otra oscilante pueden tener la misma media) y las **diferencias entre columnas de flotación**.

Construye una tabla `extra` con una fila por hora y estas columnas: `silice`, `amina_media`, `amina_std` (desviación típica de la amina dentro de la hora), `aire_medio` y el aire medio de **cada** columna de flotación (`aire_c1` … `aire_c7`). Después mira la correlación de cada una con la sílice y cuánto varía cada una de una hora a otra.

In [ ]:
# TODO: construye la tabla 'extra' y calcula la correlación de cada columna con 'silice'
#       y la desviación típica de cada columna.
#       pistas: datos.groupby('date', as_index=False)['Amina Flow'].std()
#               datos.groupby('date', as_index=False)[cols_aire].mean()
#               extra.corr()['silice']   y   extra.std()


**Pregunta:** ¿la desviación típica de la amina dice algo que no diga su media? ¿Se comportan igual las 7 columnas de flotación? Si alguna correlación sale `NaN`, ¿por qué? (Pista: mira la desviación típica de esa columna.)